In [1]:
# ============================================================
# 1. IMPORT REQUIRED LIBRARIES
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Display all dataframe columns
pd.set_option("display.max_columns", None)

# Fix the random state so results remain reproducible
RANDOM_STATE = 42

In [2]:
# ============================================================
# 2. LOAD THE FINAL DATASET
# ============================================================
# Change the path according to the location of your CSV file.

DATA_PATH = Path("../data/processed/dataset_labelled_with_score.csv")

# Example path for the uploaded dataset:
# DATA_PATH = Path("/mnt/data/dataset_labelled_with_score(2).csv")

# Check whether the dataset exists
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {DATA_PATH.resolve()}\n"
        "Please update DATA_PATH with the correct file location."
    )

# Read the CSV dataset
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

df.head()

Dataset loaded successfully.
Dataset shape: (4172, 61)


,place_id,latitude,longitude,search_area,primary_type,searched_as,label,feasibility_score,feasibility_label,cluster,bank_count_500m,bank_count_500m_normalized,bank_count_500m_contribution,bus_stop_count_500m,bus_stop_count_500m_normalized,bus_stop_count_500m_contribution,cinema_count_500m,cinema_count_500m_normalized,cinema_count_500m_contribution,clinic_count_500m,clinic_count_500m_normalized,clinic_count_500m_contribution,college_count_500m,college_count_500m_normalized,college_count_500m_contribution,hospital_count_500m,hospital_count_500m_normalized,hospital_count_500m_contribution,museum_count_500m,museum_count_500m_normalized,museum_count_500m_contribution,office_count_500m,office_count_500m_normalized,office_count_500m_contribution,parking_space_count_500m,parking_space_count_500m_normalized,parking_space_count_500m_contribution,recreation_count_500m,recreation_count_500m_normalized,recreation_count_500m_contribution,retail_count_500m,retail_count_500m_normalized,retail_count_500m_contribution,school_count_500m,school_count_500m_normalized,school_count_500m_contribution,temple_count_500m,temple_count_500m_normalized,temple_count_500m_contribution,nearest_restaurant_m,nearest_restaurant_m_normalized,nearest_restaurant_m_contribution,competitor_count_500m,competitor_count_500m_normalized,competitor_count_500m_contribution,avg_restaurant_rating_500m,avg_restaurant_rating_500m_normalized,avg_restaurant_rating_500m_contribution,avg_review_ratings_500m,avg_review_ratings_500m_normalized,avg_review_ratings_500m_contribution
0,ChIJsZ5mY70Z6zkRRBczgL9a8ms,27.699546,85.337687,Baneshwor,restaurant,restaurant,1,18.652333,Moderate,0,26,0.273684,0.018875,1,0.090909,0.007885,0,0.000000,0.000000,44,0.578947,0.026628,22,0.343750,0.023808,9,0.290323,0.012063,1,0.027778,0.00446,54,0.524272,0.022053,5,0.217391,0.016110,21,0.272727,0.009656,52,0.329114,0.007984,35,0.583333,0.017102,15,0.144231,0.010072,66.357920,0.015377,0.000851,20,0.837398,0.006476,4.265000,0.183750,0.002125,87.700000,0.941020,0.000377
1,ChIJ5SO0EQ0Z6zkRTMnqUebrTqA,27.692262,85.336472,Baneshwor,restaurant,restaurant,1,27.954012,High,2,43,0.452632,0.031216,4,0.363636,0.031538,1,0.142857,0.025274,46,0.605263,0.027838,37,0.578125,0.040041,13,0.419355,0.017424,1,0.027778,0.00446,66,0.640777,0.026953,6,0.260870,0.019332,16,0.207792,0.007357,80,0.506329,0.012284,40,0.666667,0.019545,13,0.125000,0.008729,3.883478,0.000900,0.000050,44,0.642276,0.004967,4.247727,0.188068,0.002175,159.477273,0.892192,0.000357
2,ChIJTdMmLgAZ6zkR84zXOWwdgPE,27.689064,85.334295,Baneshwor,restaurant,restaurant,1,37.665885,High,2,57,0.600000,0.041379,6,0.545455,0.047307,3,0.428571,0.075822,65,0.855263,0.039336,49,0.765625,0.053028,12,0.387097,0.016084,1,0.027778,0.00446,68,0.660194,0.027770,9,0.391304,0.028998,15,0.194805,0.006897,66,0.417722,0.010134,28,0.466667,0.013681,6,0.057692,0.004029,35.022157,0.008116,0.000449,48,0.609756,0.004716,4.231250,0.192187,0.002222,201.104167,0.863875,0.000346
3,ChIJ3Ru72S8Z6zkR6JYG3tsYlGk,27.681549,85.341340,Baneshwor,restaurant,restaurant,1,12.152369,Low,1,5,0.052632,0.003630,0,0.000000,0.000000,0,0.000000,0.000000,23,0.302632,0.013919,6,0.093750,0.006493,4,0.129032,0.005361,0,0.000000,0.00000,66,0.640777,0.026953,4,0.173913,0.012888,26,0.337662,0.011955,78,0.493671,0.011976,16,0.266667,0.007818,17,0.163462,0.011414,5.217717,0.001209,0.000067,14,0.886179,0.006854,4.371429,0.157143,0.001817,83.214286,0.944072,0.000378
4,ChIJJbL_SlQZ6zkRFWSjunPmuVU,27.688812,85.334080,Baneshwor,restaurant,restaurant,1,37.304810,High,2,55,0.578947,0.039927,6,0.545455,0.047307,3,0.428571,0.075822,66,0.868421,0.039942,46,0.718750,0.049781,11,0.354839,0.014744,1,0.027778,0.00446,65,0.631068,0.026545,10,0.434783,0.032220,15,0.194805,0.006897,64,0.405063,0.009827,27,0.450000,0.013193,7,0.067308,0.004700,35.022157,0.008116,0.000449,47,0.617886,0.004779,4.270213,0.182447,0.002110,200.893617,0.864018,0.000346


In [3]:
# ============================================================
# 3. DEFINE THE RAW MODEL FEATURES
# ============================================================
# These features describe the selected location within a
# 500-meter radius.
#
# Only features that can be collected for a new map location
# are included.
#
# EWM scores, normalized values, cluster labels and contribution
# columns are not included because they can cause data leakage.

NUMERIC_FEATURES = [
    "bank_count_500m",
    "bus_stop_count_500m",
    "cinema_count_500m",
    "clinic_count_500m",
    "college_count_500m",
    "hospital_count_500m",
    "museum_count_500m",
    "office_count_500m",
    "parking_space_count_500m",
    "recreation_count_500m",
    "retail_count_500m",
    "school_count_500m",
    "temple_count_500m",
    "nearest_restaurant_m",
    "competitor_count_500m",
    "avg_restaurant_rating_500m",
    "avg_review_ratings_500m",
]

# search_area is categorical because area names are text values
CATEGORICAL_FEATURES = [
    "search_area"
]

# Combine numerical and categorical feature names
MODEL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

# Define the target column
TARGET_COLUMN = "feasibility_label"

print("Number of numerical features:", len(NUMERIC_FEATURES))
print("Number of categorical features:", len(CATEGORICAL_FEATURES))
print("Total selected features:", len(MODEL_FEATURES))

Number of numerical features: 17
Number of categorical features: 1
Total selected features: 18


In [4]:
# ============================================================
# 4. VALIDATE REQUIRED COLUMNS
# ============================================================
# This step checks whether all selected features and the target
# column are available in the dataset.

required_columns = MODEL_FEATURES + [TARGET_COLUMN]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        "The following required columns are missing:\n"
        + "\n".join(f"- {column}" for column in missing_columns)
    )

print("All required feature and target columns are available.")

All required feature and target columns are available.


In [5]:
# ============================================================
# 5. SELECT THE RAW 500-METER FEATURES
# ============================================================
# X represents the input feature matrix.
#
# Only the selected raw location-based features are copied
# from the original dataset.

X = df[MODEL_FEATURES].copy()

print("Selected input features:")
print(X.columns.tolist())

print("\nFeature matrix shape:", X.shape)

X.head()

Selected input features:
['bank_count_500m', 'bus_stop_count_500m', 'cinema_count_500m', 'clinic_count_500m', 'college_count_500m', 'hospital_count_500m', 'museum_count_500m', 'office_count_500m', 'parking_space_count_500m', 'recreation_count_500m', 'retail_count_500m', 'school_count_500m', 'temple_count_500m', 'nearest_restaurant_m', 'competitor_count_500m', 'avg_restaurant_rating_500m', 'avg_review_ratings_500m', 'search_area']

Feature matrix shape: (4172, 18)


,bank_count_500m,bus_stop_count_500m,cinema_count_500m,clinic_count_500m,college_count_500m,hospital_count_500m,museum_count_500m,office_count_500m,parking_space_count_500m,recreation_count_500m,retail_count_500m,school_count_500m,temple_count_500m,nearest_restaurant_m,competitor_count_500m,avg_restaurant_rating_500m,avg_review_ratings_500m,search_area
0,26,1,0,44,22,9,1,54,5,21,52,35,15,66.357920,20,4.265000,87.700000,Baneshwor
1,43,4,1,46,37,13,1,66,6,16,80,40,13,3.883478,44,4.247727,159.477273,Baneshwor
2,57,6,3,65,49,12,1,68,9,15,66,28,6,35.022157,48,4.231250,201.104167,Baneshwor
3,5,0,0,23,6,4,0,66,4,26,78,16,17,5.217717,14,4.371429,83.214286,Baneshwor
4,55,6,3,66,46,11,1,65,10,15,64,27,7,35.022157,47,4.270213,200.893617,Baneshwor


In [6]:
# ============================================================
# 6. IDENTIFY EXCLUDED COLUMNS
# ============================================================
# These columns should not be used for model training.
#
# Reasons for exclusion:
# - feasibility_score was used to generate the target label
# - cluster is an output from clustering
# - normalized columns are already transformed versions
# - contribution columns are part of the EWM calculation
# - place_id and restaurant_name are identifiers
#
# Including these columns could cause data leakage.

excluded_columns = [
    column
    for column in df.columns
    if (
        column in {
            "feasibility_score",
            "cluster",
            "label",
            "place_id",
            "restaurant_name",
            "address",
        }
        or column.endswith("_normalized")
        or column.endswith("_contribution")
    )
]

print("Excluded columns:")
for column in excluded_columns:
    print(column)

Excluded columns:
place_id
label
feasibility_score
cluster
bank_count_500m_normalized
bank_count_500m_contribution
bus_stop_count_500m_normalized
bus_stop_count_500m_contribution
cinema_count_500m_normalized
cinema_count_500m_contribution
clinic_count_500m_normalized
clinic_count_500m_contribution
college_count_500m_normalized
college_count_500m_contribution
hospital_count_500m_normalized
hospital_count_500m_contribution
museum_count_500m_normalized
museum_count_500m_contribution
office_count_500m_normalized
office_count_500m_contribution
parking_space_count_500m_normalized
parking_space_count_500m_contribution
recreation_count_500m_normalized
recreation_count_500m_contribution
retail_count_500m_normalized
retail_count_500m_contribution
school_count_500m_normalized
school_count_500m_contribution
temple_count_500m_normalized
temple_count_500m_contribution
nearest_restaurant_m_normalized
nearest_restaurant_m_contribution
competitor_count_500m_normalized
competitor_count_500m_contribution

In [7]:
# ============================================================
# 7. VERIFY THAT LEAKAGE COLUMNS ARE NOT IN X
# ============================================================
# Assertions stop the notebook if a leakage column is
# accidentally included in the feature matrix.

assert "feasibility_score" not in X.columns
assert "cluster" not in X.columns
assert "label" not in X.columns
assert "place_id" not in X.columns

assert not any(
    column.endswith("_normalized")
    for column in X.columns
)

assert not any(
    column.endswith("_contribution")
    for column in X.columns
)

print("Data leakage checks passed.")

Data leakage checks passed.


In [8]:
# ============================================================
# 8. PREPARE THE TARGET COLUMN
# ============================================================
# The feasibility label is used as the prediction target.
#
# Text values are cleaned by:
# - converting to string
# - removing extra spaces
# - converting labels to title case

target_text = (
    df[TARGET_COLUMN]
    .astype("string")
    .str.strip()
    .str.title()
)

print("Unique feasibility labels:")
print(target_text.unique())

Unique feasibility labels:
<StringArray>
['Moderate', 'High', 'Low']
Length: 3, dtype: string


In [9]:
# ============================================================
# 9. VALIDATE THE FEASIBILITY LABELS
# ============================================================
# The dataset should contain only:
# Low, Moderate and High.

valid_labels = {
    "Low",
    "Moderate",
    "High"
}

actual_labels = set(
    target_text.dropna().unique()
)

invalid_labels = actual_labels - valid_labels

if invalid_labels:
    raise ValueError(
        f"Invalid feasibility labels found: {invalid_labels}"
    )

if target_text.isna().any():
    missing_target_count = target_text.isna().sum()

    raise ValueError(
        f"{missing_target_count} target values are missing."
    )

print("Feasibility labels are valid.")

Feasibility labels are valid.


In [10]:
# ============================================================
# 10. ENCODE THE TARGET LABELS
# ============================================================
# Machine learning models require numerical target values.
#
# Encoding:
# Low      = 0
# Moderate = 1
# High     = 2

LABEL_MAPPING = {
    "Low": 0,
    "Moderate": 1,
    "High": 2,
}

# Reverse mapping will later convert numerical predictions
# back into readable class names.
INVERSE_LABEL_MAPPING = {
    0: "Low",
    1: "Moderate",
    2: "High",
}

# Encode the target
y = target_text.map(LABEL_MAPPING).astype("int64")

print("Target encoding completed.")
print(y.head())

Target encoding completed.
0    1
1    2
2    2
3    0
4    2
Name: feasibility_label, dtype: int64


In [11]:
# ============================================================
# 11. DISPLAY THE TARGET DISTRIBUTION
# ============================================================
# This table shows the number and percentage of observations
# belonging to each feasibility class.

target_distribution = pd.DataFrame({
    "count": target_text.value_counts(),
    "percentage": (
        target_text.value_counts(normalize=True) * 100
    ).round(2),
})

target_distribution

,count,percentage
feasibility_label,,
Low,1671,40.05
Moderate,1644,39.41
High,857,20.54


In [12]:
# ============================================================
# 12. CONVERT NUMERICAL FEATURES TO NUMERIC DATA TYPE
# ============================================================
# Any invalid text value in a numerical feature is converted
# to NaN.
#
# Missing numerical values will later be replaced using
# median imputation.

for column in NUMERIC_FEATURES:
    X[column] = pd.to_numeric(
        X[column],
        errors="coerce"
    )

print("Numerical feature conversion completed.")

Numerical feature conversion completed.


In [13]:
# ============================================================
# 13. CHECK MISSING VALUES BEFORE PREPROCESSING
# ============================================================
# Missing values are not filled using the complete dataset.
#
# Instead, imputation values will be learned only from the
# training dataset to prevent data leakage.

missing_values = (
    X.isna()
    .sum()
    .sort_values(ascending=False)
)

missing_values = missing_values[
    missing_values > 0
]

if missing_values.empty:
    print("No missing feature values were found.")
else:
    print("Missing feature values:")
    display(
        missing_values.to_frame("missing_count")
    )

No missing feature values were found.


In [14]:
# ============================================================
# 14. CHECK THE CATEGORICAL FEATURE
# ============================================================
# Display the available search areas and their frequencies.

print("Unique search areas:")
print(X["search_area"].unique())

print("\nSearch area distribution:")
print(X["search_area"].value_counts(dropna=False))

Unique search areas:
['Baneshwor' 'New Road' 'Koteshwor' 'Bhaktapur durbar square'
 'Patan durbar square' 'Boudha stupa' 'Pulchowk' 'Durbar Marg' 'Kirtipur']

Search area distribution:
search_area
Baneshwor                  532
Bhaktapur durbar square    518
New Road                   517
Patan durbar square        516
Boudha stupa               511
Kirtipur                   425
Durbar Marg                405
Koteshwor                  401
Pulchowk                   347
Name: count, dtype: int64


In [15]:
# ============================================================
# 15. PERFORM A STRATIFIED TRAIN-TEST SPLIT
# ============================================================
# The dataset is divided into:
# - 80% training data
# - 20% testing data
#
# stratify=y ensures that Low, Moderate and High classes keep
# approximately the same proportions in both subsets.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)

print("Training target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)

Training feature shape: (3337, 18)
Testing feature shape: (835, 18)
Training target shape: (3337,)
Testing target shape: (835,)


In [16]:
# ============================================================
# 16. VERIFY CLASS DISTRIBUTION AFTER SPLITTING
# ============================================================
# The percentages should remain approximately similar in the
# full dataset, training set and testing set.

split_distribution = pd.DataFrame({
    "Full dataset (%)": (
        y.value_counts(normalize=True).sort_index() * 100
    ),
    "Training set (%)": (
        y_train.value_counts(normalize=True).sort_index() * 100
    ),
    "Testing set (%)": (
        y_test.value_counts(normalize=True).sort_index() * 100
    ),
}).round(2)

split_distribution.index = [
    INVERSE_LABEL_MAPPING[index]
    for index in split_distribution.index
]

split_distribution

,Full dataset (%),Training set (%),Testing set (%)
Low,40.05,40.07,40.0
Moderate,39.41,39.41,39.4
High,20.54,20.53,20.6


In [17]:
# ============================================================
# 17. CREATE THE NUMERICAL PREPROCESSING PIPELINE
# ============================================================
# Numerical preprocessing contains two steps:
#
# 1. SimpleImputer
#    Replaces missing values with the median value learned
#    from the training dataset.
#
# 2. StandardScaler
#    Standardizes each numerical feature using:
#
#    z = (x - mean) / standard deviation
#
# Standardization is especially important for models such as
# Logistic Regression.

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
    ]
)

numeric_transformer

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])

In [18]:
# ============================================================
# 18. CREATE THE CATEGORICAL PREPROCESSING PIPELINE
# ============================================================
# Categorical preprocessing contains two steps:
#
# 1. Replace missing search_area values using the most
#    frequently occurring training value.
#
# 2. Convert search_area into separate binary columns using
#    one-hot encoding.
#
# handle_unknown="ignore" prevents errors if a new area appears
# during testing or prediction.

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        ),
    ]
)

categorical_transformer

Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot',
                 OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

In [19]:
# ============================================================
# 19. COMBINE NUMERICAL AND CATEGORICAL PREPROCESSING
# ============================================================
# ColumnTransformer applies:
#
# - numeric_transformer to NUMERIC_FEATURES
# - categorical_transformer to CATEGORICAL_FEATURES
#
# remainder="drop" removes any columns that are not explicitly
# included in the transformer.

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            NUMERIC_FEATURES
        ),
        (
            "categorical",
            categorical_transformer,
            CATEGORICAL_FEATURES
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

preprocessor

ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['bank_count_500m', 'bus_stop_count_500m',
                                  'cinema_count_500m', 'clinic_count_500m',
                                  'college_count_500m', 'hospital_count_500m',
                                  'museum_count_500m', 'office_count_500m',
                                  'parking_space_count_500m',
                                  'recreation_count_500m', '...,
                                  'school_count_500m', 'temple_count_500m',
                                  'nearest_restaurant_m',
                                  'competitor_count_500m',
                                  'avg_restaurant_rating_500m',
                                  'avg_review_ratings_500m']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['search_area'])],
                  verbose_feature_names_out=False)

In [20]:
# ============================================================
# 20. FIT THE PREPROCESSOR ON TRAINING DATA
# ============================================================
# The preprocessor must be fitted only on X_train.
#
# This ensures that:
# - medians are calculated only from training data
# - means and standard deviations are learned only from
#   training data
# - categories are learned only from training data
#
# This prevents information from the test dataset from leaking
# into model training.

X_train_processed = preprocessor.fit_transform(
    X_train
)

print("Preprocessor fitted on training data.")
print(
    "Processed training data shape:",
    X_train_processed.shape
)

Preprocessor fitted on training data.
Processed training data shape: (3337, 26)


In [21]:
# ============================================================
# 21. TRANSFORM THE TEST DATA
# ============================================================
# The already-fitted preprocessor is applied to X_test.
#
# Do not call fit_transform on the test dataset.

X_test_processed = preprocessor.transform(
    X_test
)

print(
    "Processed testing data shape:",
    X_test_processed.shape
)

Processed testing data shape: (835, 26)


In [22]:
# ============================================================
# 22. GET THE FINAL TRANSFORMED FEATURE NAMES
# ============================================================
# The final dataset contains:
# - normalized numerical features
# - one-hot encoded search-area columns

processed_feature_names = (
    preprocessor.get_feature_names_out()
)

print("Final transformed features:")

for feature in processed_feature_names:
    print(feature)

Final transformed features:
bank_count_500m
bus_stop_count_500m
cinema_count_500m
clinic_count_500m
college_count_500m
hospital_count_500m
museum_count_500m
office_count_500m
parking_space_count_500m
recreation_count_500m
retail_count_500m
school_count_500m
temple_count_500m
nearest_restaurant_m
competitor_count_500m
avg_restaurant_rating_500m
avg_review_ratings_500m
search_area_Baneshwor
search_area_Bhaktapur durbar square
search_area_Boudha stupa
search_area_Durbar Marg
search_area_Kirtipur
search_area_Koteshwor
search_area_New Road
search_area_Patan durbar square
search_area_Pulchowk


In [23]:
# ============================================================
# 23. CONVERT PROCESSED ARRAYS INTO DATAFRAMES
# ============================================================
# Converting the processed arrays into DataFrames makes them
# easier to inspect and understand.

X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=processed_feature_names,
    index=X_train.index,
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=processed_feature_names,
    index=X_test.index,
)

print("Processed training dataframe:")
display(X_train_processed_df.head())

print("Processed testing dataframe:")
display(X_test_processed_df.head())

Processed training dataframe:


,bank_count_500m,bus_stop_count_500m,cinema_count_500m,clinic_count_500m,college_count_500m,hospital_count_500m,museum_count_500m,office_count_500m,parking_space_count_500m,recreation_count_500m,retail_count_500m,school_count_500m,temple_count_500m,nearest_restaurant_m,competitor_count_500m,avg_restaurant_rating_500m,avg_review_ratings_500m,search_area_Baneshwor,search_area_Bhaktapur durbar square,search_area_Boudha stupa,search_area_Durbar Marg,search_area_Kirtipur,search_area_Koteshwor,search_area_New Road,search_area_Patan durbar square,search_area_Pulchowk
2824,1.842231,0.419158,-0.591109,2.534915,1.175470,4.000334,0.061296,0.851153,0.566327,-0.059980,1.485149,1.049286,1.102903,-0.366214,0.031513,-0.843309,-0.247755,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2262,-1.005569,-0.034239,-0.591109,-1.123237,-0.983523,-1.177397,-0.548828,-1.188945,-0.746514,-1.165323,-0.461526,-1.202281,-0.763273,0.213421,-0.861790,2.231056,-1.194338,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2278,-0.950804,-0.034239,-0.591109,-0.503211,-0.638084,-0.807559,-0.548828,-0.632554,-0.746514,-1.080297,-0.785972,-0.357943,-0.571871,0.121484,-0.647397,0.882764,0.003195,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2796,-0.622212,-0.487637,-0.591109,-0.751221,1.693629,-0.807559,1.281543,1.036617,0.566327,0.110073,0.349589,-0.357943,0.098039,-0.299592,-0.075683,0.374713,0.435638,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
690,0.144504,0.872556,-0.591109,0.178817,-0.638084,0.301955,-0.548828,-0.855110,-0.746514,-0.315059,0.609145,0.580210,-0.476169,-0.783424,-0.218612,1.100008,-0.717398,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Processed testing dataframe:


,bank_count_500m,bus_stop_count_500m,cinema_count_500m,clinic_count_500m,college_count_500m,hospital_count_500m,museum_count_500m,office_count_500m,parking_space_count_500m,recreation_count_500m,retail_count_500m,school_count_500m,temple_count_500m,nearest_restaurant_m,competitor_count_500m,avg_restaurant_rating_500m,avg_review_ratings_500m,search_area_Baneshwor,search_area_Bhaktapur durbar square,search_area_Boudha stupa,search_area_Durbar Marg,search_area_Kirtipur,search_area_Koteshwor,search_area_New Road,search_area_Patan durbar square,search_area_Pulchowk
4028,-1.005569,-0.487637,-0.591109,-1.309245,1.261830,-1.177397,-0.548828,-1.300223,-0.965320,-0.485112,-1.499752,-1.483726,-0.811124,1.554166,-0.897522,-0.066902,-0.250086,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
235,2.335120,0.419158,0.378670,1.170858,0.139154,0.856712,2.806853,1.222080,1.222748,2.235734,3.172267,0.580210,1.820663,-0.795375,2.139708,0.421380,1.406507,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
113,-0.841273,0.419158,-0.591109,-0.069193,-0.724443,1.041631,-0.243766,-0.076164,-0.527707,-0.740191,-1.110417,-1.014650,-0.667572,-0.495111,-0.575933,-0.739093,0.297538,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
910,2.006527,2.232748,-0.591109,2.720923,2.125427,2.151144,-0.548828,1.741378,0.128714,0.280126,1.550038,2.081254,0.385143,-0.781023,0.353102,-0.099143,-0.098219,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
318,3.156600,-0.941034,1.348450,1.108856,-0.724443,1.041631,1.129012,0.925339,2.535589,0.195100,0.446922,0.486394,3.160481,-0.865851,1.246405,-1.240975,0.477315,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [24]:
# ============================================================
# 24. CHECK FOR MISSING VALUES AFTER PREPROCESSING
# ============================================================
# No missing values should remain after median and
# most-frequent imputation.

training_missing = (
    X_train_processed_df
    .isna()
    .sum()
    .sum()
)

testing_missing = (
    X_test_processed_df
    .isna()
    .sum()
    .sum()
)

print(
    "Missing values in processed training data:",
    training_missing
)

print(
    "Missing values in processed testing data:",
    testing_missing
)

if training_missing == 0 and testing_missing == 0:
    print("No missing values remain after preprocessing.")
else:
    raise ValueError(
        "Missing values remain after preprocessing."
    )

Missing values in processed training data: 0
Missing values in processed testing data: 0
No missing values remain after preprocessing.


In [25]:
# ============================================================
# 25. VERIFY NORMALIZED NUMERICAL FEATURES
# ============================================================
# Numerical training columns should have a mean approximately
# equal to zero after StandardScaler.
#
# Small differences such as 0.000001 are normal because of
# floating-point calculations.

normalized_feature_means = (
    X_train_processed_df[NUMERIC_FEATURES]
    .mean()
    .round(6)
)

normalized_feature_means.to_frame(
    "training_mean"
)

,training_mean
bank_count_500m,0.0
bus_stop_count_500m,0.0
cinema_count_500m,-0.0
clinic_count_500m,0.0
college_count_500m,0.0
hospital_count_500m,-0.0
museum_count_500m,0.0
office_count_500m,-0.0
parking_space_count_500m,0.0
recreation_count_500m,0.0


In [26]:
# ============================================================
# 26. DISPLAY ONE-HOT ENCODED AREA COLUMNS
# ============================================================
# Each search area is represented by a separate binary column.

encoded_area_columns = [
    column
    for column in processed_feature_names
    if column.startswith("search_area_")
]

print("One-hot encoded search-area columns:")

for column in encoded_area_columns:
    print(column)

One-hot encoded search-area columns:
search_area_Baneshwor
search_area_Bhaktapur durbar square
search_area_Boudha stupa
search_area_Durbar Marg
search_area_Kirtipur
search_area_Koteshwor
search_area_New Road
search_area_Patan durbar square
search_area_Pulchowk


In [27]:
# ============================================================
# 27. FINAL PHASE 3 SUMMARY
# ============================================================
# The following objects are now ready for Phase 4:
#
# X_train
# X_test
# y_train
# y_test
# preprocessor
# NUMERIC_FEATURES
# CATEGORICAL_FEATURES
# LABEL_MAPPING
# INVERSE_LABEL_MAPPING
#
# During model development, the preprocessor should be placed
# inside the same Pipeline as the classifier.
#
# This ensures that preprocessing and prediction are performed
# consistently during training, evaluation and deployment.

print("Phase 3 completed successfully.")

print("\nRaw training shape:", X_train.shape)
print("Raw testing shape:", X_test.shape)

print(
    "Processed training shape:",
    X_train_processed_df.shape
)

print(
    "Processed testing shape:",
    X_test_processed_df.shape
)

Phase 3 completed successfully.

Raw training shape: (3337, 18)
Raw testing shape: (835, 18)
Processed training shape: (3337, 26)
Processed testing shape: (835, 26)


# Model development 